# Modelo de Fluxo de Tráfego

Este notebook cria um modelo para classificar o `nivel_de_fluxo_15min` (baixo, moderado, alto)
na próxima janela de 15 minutos por radar/sentido/faixa.

- Dataset de origem: `producer/radar/dataset/20230901`
- Abordagem: agregação temporal + classificador supervisionado


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATASET_DIR = Path('../../../producer/radar/dataset/20230901').resolve()
MODEL_PATH = Path('traffic_flow_model.joblib').resolve()
DATASET_DIR, MODEL_PATH


(PosixPath('/home/makleyston/Projects/service-composition/producer/radar/dataset/20230901'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-flow/traffic_flow_model.joblib'))

In [9]:
json_files = sorted(DATASET_DIR.glob('*.json'))
len(json_files), json_files[0].name, json_files[-1].name


(24, '20230901_00.json', '20230901_23.json')

In [10]:
frames = []
use_cols = ['ID EQP', 'DATA HORA', 'FAIXA', 'SENTIDO', 'VELOCIDADE AFERIDA', 'CLASSIFICAÇÃO']

for file_path in json_files:
    raw = pd.read_json(file_path)
    df_part = raw[use_cols].copy()

    df_part['DATA HORA'] = pd.to_datetime(df_part['DATA HORA'], errors='coerce')
    df_part['ID EQP'] = pd.to_numeric(df_part['ID EQP'], errors='coerce')
    df_part['FAIXA'] = pd.to_numeric(df_part['FAIXA'], errors='coerce')
    df_part['VELOCIDADE AFERIDA'] = pd.to_numeric(df_part['VELOCIDADE AFERIDA'], errors='coerce')

    df_part = df_part.dropna(subset=['DATA HORA', 'ID EQP', 'FAIXA'])
    df_part['window_start'] = df_part['DATA HORA'].dt.floor('15min')

    grouped = (
        df_part
        .groupby(['ID EQP', 'SENTIDO', 'FAIXA', 'window_start'], as_index=False)
        .agg(
            vehicle_count=('ID EQP', 'size'),
            avg_speed=('VELOCIDADE AFERIDA', 'mean'),
            speed_std=('VELOCIDADE AFERIDA', 'std'),
            pct_moto=('CLASSIFICAÇÃO', lambda s: (s == 'MOTO').mean()),
            pct_heavy=('CLASSIFICAÇÃO', lambda s: s.astype(str).str.contains('CAMINHÃO|ÔNIBUS', na=False).mean()),
        )
    )
    frames.append(grouped)

df_agg = pd.concat(frames, ignore_index=True)
df_agg = (
    df_agg
    .groupby(['ID EQP', 'SENTIDO', 'FAIXA', 'window_start'], as_index=False)
    .agg(
        vehicle_count=('vehicle_count', 'sum'),
        avg_speed=('avg_speed', 'mean'),
        speed_std=('speed_std', 'mean'),
        pct_moto=('pct_moto', 'mean'),
        pct_heavy=('pct_heavy', 'mean'),
    )
)

df_agg = df_agg.sort_values(['ID EQP', 'SENTIDO', 'FAIXA', 'window_start']).reset_index(drop=True)
df_agg['hour'] = df_agg['window_start'].dt.hour
df_agg['weekday'] = df_agg['window_start'].dt.dayofweek

df_agg.head()


,ID EQP,SENTIDO,FAIXA,window_start,vehicle_count,avg_speed,speed_std,pct_moto,pct_heavy,hour,weekday
0,182,Jatobá/Milionários,3,2023-09-01 00:00:00,28,45.678571,5.277861,0.071429,0.071429,0,4
1,182,Jatobá/Milionários,3,2023-09-01 00:15:00,26,47.461538,7.895471,0.307692,0.115385,0,4
2,182,Jatobá/Milionários,3,2023-09-01 00:30:00,15,47.266667,5.297798,0.133333,0.133333,0,4
3,182,Jatobá/Milionários,3,2023-09-01 00:45:00,19,43.894737,5.486293,0.052632,0.157895,0,4
4,182,Jatobá/Milionários,3,2023-09-01 01:00:00,15,44.400000,7.604510,0.066667,0.000000,1,4


In [11]:
key_cols = ['ID EQP', 'SENTIDO', 'FAIXA']
df_agg['target_count_next_15m'] = df_agg.groupby(key_cols)['vehicle_count'].shift(-1)

df_model = df_agg.dropna(subset=['target_count_next_15m']).copy()

q1 = df_model['target_count_next_15m'].quantile(0.33)
q2 = df_model['target_count_next_15m'].quantile(0.66)

df_model['nivel_de_fluxo_15min'] = np.select(
    [
        df_model['target_count_next_15m'] <= q1,
        (df_model['target_count_next_15m'] > q1) & (df_model['target_count_next_15m'] <= q2),
        df_model['target_count_next_15m'] > q2,
    ],
    ['baixo', 'moderado', 'alto'],
    default='moderado',
)

df_model['nivel_de_fluxo_15min'].value_counts(normalize=True).rename('proporcao')


nivel_de_fluxo_15min
alto        0.338829
moderado    0.330761
baixo       0.330410
Name: proporcao, dtype: float64

In [12]:
feature_cols = [
    'ID EQP',
    'SENTIDO',
    'FAIXA',
    'vehicle_count',
    'avg_speed',
    'speed_std',
    'pct_moto',
    'pct_heavy',
    'hour',
    'weekday',
]
target_col = 'nivel_de_fluxo_15min'

X = df_model[feature_cols]
y = df_model[target_col]

numeric_features = ['vehicle_count', 'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy', 'hour', 'weekday']
categorical_features = ['ID EQP', 'SENTIDO', 'FAIXA']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
print('Matriz de confusao:')
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

        alto       0.91      0.87      0.89      1352
       baixo       0.94      0.92      0.93      1319
    moderado       0.81      0.86      0.83      1320

    accuracy                           0.88      3991
   macro avg       0.89      0.88      0.88      3991
weighted avg       0.89      0.88      0.88      3991

Matriz de confusao:
[[1182    6  164]
 [   3 1210  106]
 [ 113   72 1135]]


In [13]:
preview = X_test.head(5).copy()
preview['nivel_real'] = y_test.head(5).values
preview['nivel_predito'] = clf.predict(X_test.head(5))
preview


,ID EQP,SENTIDO,FAIXA,vehicle_count,avg_speed,speed_std,pct_moto,pct_heavy,hour,weekday,nivel_real,nivel_predito
19956,254,Bairro/Centro,2,236,46.025424,5.229966,0.072034,0.156780,14,4,alto,alto
12725,226,Sta. Efigênia / Savassi,4,150,28.560000,12.177920,0.113333,0.086667,14,4,moderado,moderado
16729,239,Bairro/Centro,1,15,49.266667,6.273148,0.000000,0.133333,0,4,baixo,baixo
13955,230,Betânia/Barreiro,1,126,47.023810,6.205113,0.206349,0.126984,17,4,moderado,moderado
13137,227,Centro/Bairro,2,72,28.458333,15.549954,0.125000,0.041667,21,4,moderado,moderado


In [14]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'labeling_rule': {
        'baixo': f'target_count_next_15m <= {q1:.2f}',
        'moderado': f'{q1:.2f} < target_count_next_15m <= {q2:.2f}',
        'alto': f'target_count_next_15m > {q2:.2f}',
    },
    'window_minutes': 15,
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH


PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-flow/traffic_flow_model.joblib')